# Synthetic data for the CLOAK methods paper

This notebook writes every synthetic input the paper uses, into this directory:

1. **flux-vs-nH tables**, one per Chandra band, from a *fake* absorbed power-law
   spectrum (`cloak.synthetic.spectrum` + `cloak.flux_table`, needs HEASoft/PyXspec
   and a response pair);
2. **synthetic light curves** of the two fiducial systems defined in
   `cloak/synthetic/fiducial.py` (System A, WR-like; System B, OB-like), in every
   band with a table, with a truth record next to each file;
3. diagnostic plots of what was generated (the paper's *figure* code lives in
   `figures/paper_figures.ipynb`, not here).

Everything is deterministic (fixed seeds) and skips outputs that already exist;
set `FORCE = True` to regenerate. Run from any directory.

In [ ]:
import json, os, subprocess, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

HERE = os.path.abspath(os.getcwd())
ROOT = HERE if os.path.isdir(os.path.join(HERE, "cloak")) else os.path.dirname(HERE)
sys.path.insert(0, ROOT)
from cloak import kernel as K
from cloak import utils as U
from cloak.synthetic import fiducial as F

PY = sys.executable
DATA = os.path.join(ROOT, "synthetic_data")
TABLES = os.path.join(DATA, "tables")
BANDS = dict(U.CHANDRA_BANDS)
FORCE = False
print("repository:", ROOT)

## Configuration

The fake spectrum is a generic absorbed power law (not IC 10 X-1's fit). The
response pair is instrument calibration and is **not** distributed: point `RMF`
and `ARF` at any Chandra ACIS (or other) RMF/ARF you have. Without HEASoft the
table step is skipped and whatever tables exist under `synthetic_data/` are used
(the tracked example `flux_vs_nH_tbabs_broad.csv` at least).

In [ ]:
RMF = os.environ.get("CLOAK_RMF", os.path.join(ROOT, "data", "IC10X1_spec", "X1_spectrum_combined_src.rmf"))
ARF = os.environ.get("CLOAK_ARF", os.path.join(ROOT, "data", "IC10X1_spec", "X1_spectrum_combined_src.arf"))
SPECTRUM = dict(model="tbabs", nH=0.3, PhoIndex=1.8, norm=1.0e-3, exposure=1.0e5, seed=1)   # nH in 1e22 cm^-2
TABLE_GRID = dict(nH_min=1e19, nH_max=1e26, nH_points=600)

def have_xspec():
    try:
        import xspec  # noqa: F401
        return True
    except Exception:
        return False

print("PyXspec available:", have_xspec(), "| responses present:", os.path.exists(RMF) and os.path.exists(ARF))

## 1. Flux-vs-nH tables (HEASoft)

In [ ]:
os.makedirs(TABLES, exist_ok=True)
spec_dir = os.path.join(DATA, "spec")
if have_xspec() and os.path.exists(RMF) and os.path.exists(ARF):
    if FORCE or not os.path.exists(os.path.join(spec_dir, "synthetic_src.pha")):
        subprocess.run([PY, "-m", "cloak.synthetic.spectrum", "--out-dir", spec_dir, "--rmf", RMF, "--arf", ARF,
                        "--model", SPECTRUM["model"], "--nH", str(SPECTRUM["nH"]), "--PhoIndex", str(SPECTRUM["PhoIndex"]),
                        "--norm", str(SPECTRUM["norm"]), "--exposure", str(SPECTRUM["exposure"]), "--seed", str(SPECTRUM["seed"])],
                       cwd=ROOT, check=True)
    for band in BANDS:
        out_csv = os.path.join(TABLES, f"flux_vs_nH_{band}.csv")
        if FORCE or not os.path.exists(out_csv):
            subprocess.run([PY, "-m", "cloak.flux_table", "--specdir", spec_dir, "--model", SPECTRUM["model"], "--band", band,
                            "--nH_min", str(TABLE_GRID["nH_min"]), "--nH_max", str(TABLE_GRID["nH_max"]),
                            "--nH_points", str(TABLE_GRID["nH_points"]), "--out_csv", out_csv,
                            "--out_png", out_csv.replace(".csv", ".png")], cwd=ROOT, check=True)
else:
    print("Skipping the table step (needs PyXspec: run `heainit` first, and a response pair in RMF/ARF).")

def table_for(band):
    for cand in (os.path.join(TABLES, f"flux_vs_nH_{band}.csv"), os.path.join(DATA, f"flux_vs_nH_tbabs_{band}.csv")):
        if os.path.exists(cand):
            return cand
    return None

tables = {b: table_for(b) for b in BANDS if table_for(b)}
print("tables in use:"); [print(f"  {b:7s} {os.path.relpath(p, ROOT)}") for b, p in tables.items()];

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for band, path in tables.items():
    df, _ = K.load_flux_vs_nh_csv(path, "erg")
    ax.loglog(df["nH_1e22"], df[f"flux_{band}_erg"], label=f"{band} ({BANDS[band][0]:g}-{BANDS[band][1]:g} keV)")
ax.set(xlabel=r"$N_{\rm H}$ ($10^{22}$ cm$^{-2}$)", ylabel="band flux (erg cm$^{-2}$ s$^{-1}$)", title="flux vs nH tables in use")
ax.legend(); plt.show()

## 2. Fiducial systems

In [ ]:
rows = []
for name, s in F.SYSTEMS.items():
    d = F.describe(s); g = F.geometry(s)
    rows.append({"system": name, **{k: g[k] for k in ("d1", "d2", "R", "r", "i0")}, "a": d["a"], "M_tot": round(d["M_tot"], 2),
                 "P (d)": d["period_d"], "profile": s["wind_model"], **s["wind_params"], "mdot": s["mdot"], "v_inf": s["v_inf"],
                 "f_opa": s["f_opacity"], "total eclipse": d["total_eclipse"]})
pd.DataFrame(rows).set_index("system").T

In [ ]:
# Model light curves at dth = 1 in every band with a table (normalized), as a sanity check of the systems.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (name, s) in zip(axes, F.SYSTEMS.items()):
    for band, path in tables.items():
        ph, fl = K.simulate_band_flux(**F.simulation_kwargs(s), flux_csv_path=path, band=band, dth=1.0)
        ax.plot(ph, fl / fl.max(), label=band)
    ax.set(title=s["label"], xlabel="orbital phase", xlim=(0.3, 0.7)); ax.legend()
axes[0].set_ylabel(r"$F_b / F_{b,\rm out}$"); plt.show()

## 3. Synthetic light curves

For every system and band: the out-of-eclipse model flux sets the count rate
(`target_rate`) and the scattered floor (`scatter_fraction`), and
`cloak.synthetic.lightcurve` draws Poisson counts over the visits with gaps,
writing the CIAO layout plus an `exposure` column and `<stem>_truth.json`.
System A carries 10 % intrinsic log-normal variability so the jitter likelihood
has something to measure; its light curve is shifted by 0.02 in phase.

In [ ]:
def generate(name, band, force=FORCE):
    s = F.SYSTEMS[name]; obs = s["observation"]
    out_dir = os.path.join(DATA, f"system{name}", band)
    out = os.path.join(out_dir, f"system{name}_{band}.txt")
    if os.path.exists(out) and not force:
        return out
    kw = F.simulation_kwargs(s)
    ph, fl = K.simulate_band_flux(**kw, flux_csv_path=tables[band], band=band, dth=1.0)
    f_out = float(fl.max())
    floor = obs["scatter_fraction"] * f_out
    flux_per_rate = f_out / obs["target_rate"]
    visits = ",".join(f"{a:g}:{d:g}" for a, d in obs["visits"])
    cmd = [PY, "-m", "cloak.synthetic.lightcurve", "--flux-csv", tables[band], "--band", band,
           "--r", str(kw["r"]), "--R", str(kw["R"]), "--d1", str(kw["d1"]), "--d2", str(kw["d2"]), "--i0", str(kw["i0"]),
           "--wind-model", kw["wind_model"], "--mdot", str(kw["mdot"]), "--v-inf", str(kw["v_inf"]), "--mu-wind", str(kw["mu_wind"]),
           "--f-opacity", str(kw["f_opacity"]), "--dth", "1.0", "--orbital-period", str(s["period_s"]),
           "--phase-shift", str(obs["phase_shift"]), "--scatter", f"{floor:.6e}", "--intrinsic-scatter", str(obs["intrinsic_scatter"]),
           "--flux-per-rate", f"{flux_per_rate:.6e}", "--dt", str(obs["dt"]), "--visits", visits,
           "--gap-fraction", str(obs["gap_fraction"]), "--gap-duration", str(obs["gap_duration"]),
           "--seed", str(obs["seed"]), "--output", out]
    for key, val in kw["wind_params"].items():
        if key != "R_star":
            cmd += [f"--{key}", str(val)]
    subprocess.run(cmd, cwd=ROOT, check=True, capture_output=True, text=True)
    return out

generated = {}
for name in F.SYSTEMS:
    for band in tables:
        generated[(name, band)] = generate(name, band)
        print(f"system {name} {band:7s} -> {os.path.relpath(generated[(name, band)], ROOT)}")

## 4. Diagnostics of the generated data

In [ ]:
for name in F.SYSTEMS:
    bands = [b for (n, b) in generated if n == name]
    fig, axes = plt.subplots(1, len(bands), figsize=(4.2 * len(bands), 3.4), squeeze=False)
    for ax, band in zip(axes[0], bands):
        df = U.load_data(os.path.dirname(generated[(name, band)]), obs_column="flux_t", time_column="t_raw",
                         period=F.SYSTEMS[name]["period_s"])
        binned = U.phase_bin_data_snr(df, counts_per_bin=100, verbose=False)
        with open(generated[(name, band)].replace(".txt", "_truth.json")) as fh:
            truth = json.load(fh)
        ax.plot(df["phase"], df["rate"], ".", ms=2, color="0.7", label="100 s bins")
        ax.errorbar(binned["phase"], binned["rate"], yerr=binned["error"], xerr=0.5 * binned["width"], fmt="o", ms=3, color="C0",
                    label="100-count bins (exposure-weighted)")
        ax.axvline(truth["mid_eclipse_data_phase"], color="crimson", lw=0.8, ls="--", label="injected mid-eclipse")
        ax.set(title=f"System {name}, {band}: {truth['total_counts']:.0f} counts, {truth['n_bins']} bins, "
                     f"{truth['zero_count_bins']} empty", xlabel="orbital phase", ylabel="flux (erg cm$^{-2}$ s$^{-1}$)")
        ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

In [ ]:
# Time coverage of System A: visits, gaps and how many orbits they span.
df = U.load_data(os.path.dirname(generated[("A", "broad")]), obs_column="flux_t", time_column="t_raw", period=F.SYSTEMS["A"]["period_s"])
t_days = (df["time"] - df["time"].min()) / 86400.0
fig, ax = plt.subplots(figsize=(9, 2.2))
ax.plot(t_days, df["rate"], ".", ms=2)
ax.set(xlabel="days since first bin", ylabel="flux", title=f"System A broad: {len(df)} bins over {t_days.max():.1f} d "
       f"({t_days.max() * 86400 / F.SYSTEMS['A']['period_s']:.1f} orbits)")
plt.show()

## 5. What was written

Tracked outputs (git re-includes `*.csv`, `*.txt` and `*.json` under `synthetic_data/`):
the tables in `tables/`, and per system and band `system<X>/<band>/system<X>_<band>.txt`
with `..._truth.json`. The paper-figure notebook reads exactly these paths.

In [ ]:
for root, _, files in os.walk(DATA):
    for f in sorted(files):
        if f.endswith((".csv", ".txt", ".json")):
            p = os.path.join(root, f); print(f"{os.path.getsize(p) / 1e3:8.1f} kB  {os.path.relpath(p, ROOT)}")